# 🦀 Day 6: CLI or API Interface
---

Today we build the user-facing interface for RustKV.

- **clap**: Command-line parsing with `derive` feature
- **Server mode**: `rustkv server --port 6379 --data-dir ./data`
- **Client mode**: `rustkv client --host 127.0.0.1 --port 6379`
- **Interactive REPL**: Read commands, send, display results
- **Colored output**: Success in green, errors in red
- **Configuration**: CLI args > env vars > config file > defaults
- **Exercise**: Add `--format` flag for JSON or plain text output

## Add clap dependency

```toml
[dependencies]
clap = { version = "4", features = ["derive"] }
```

In [ ]:
:dep clap = { version = "4", features = ["derive"] }

use clap::{Parser, Subcommand};

#[derive(Parser, Debug)]
#[command(name = "rustkv", version, about)]
struct Cli {
    #[command(subcommand)]
    command: Commands,
}

#[derive(Subcommand, Debug)]
enum Commands {
    Server {
        #[arg(short, long, default_value = "6379")]
        port: u16,
        #[arg(long, default_value = "./data")]
        data_dir: String,
    },
    Client {
        #[arg(short, long, default_value = "127.0.0.1")]
        host: String,
        #[arg(short, long, default_value = "6379")]
        port: u16,
    },
}

let mut args = vec!["rustkv", "server", "--port", "6380"];
let cli = Cli::parse_from(args);
println!("Parsed: {:?}", cli);

In [ ]:
// REPL loop structure

let repl_code = r#"
fn run_repl(host: &str, port: u16) -> Result<(), Box<dyn std::error::Error>> {
    let mut stream = TcpStream::connect(format!("{}:{}", host, port))?;
    let mut rdr = std::io::BufReader::new(&stream);
    let mut wtr = std::io::BufWriter::new(&stream);
    let mut line = String::new();

    loop {
        print!("rustkv> ");
        std::io::stdout().flush()?;
        line.clear();
        if std::io::stdin().read_line(&mut line)? == 0 { break; }
        let cmd = line.trim();
        if cmd.is_empty() { continue; }
        if cmd.eq_ignore_ascii_case("quit") { break; }

        wtr.write_all(cmd.as_bytes())?;
        wtr.write_all(b"\n")?;
        wtr.flush()?;

        let mut buf = String::new();
        rdr.read_line(&mut buf)?;
        println!("{}", buf.trim());
    }
    Ok(())
}
"#;

println!("REPL loop:");
println!("{}", repl_code);

In [ ]:
// Colored output (add colored or owo-colors for colored terminal output)

let colored_usage = r#"
// In Cargo.toml: owo-colors = "4"
// use owo_colors::OwoColorize;

// Success: println!("{}", response.green());
// Error:   println!("{}", response.red());
// Or match on response: if response.starts_with("-ERR") { red } else { green }
"#;

println!("Colored output:");
println!("{}", colored_usage);

In [ ]:
// main.rs dispatch — match on parsed CLI

let main_code = r#"
fn main() -> Result<(), Box<dyn std::error::Error>> {
    let cli = Cli::parse();
    match cli.command {
        Commands::Server { port, data_dir } => {
            println!("Starting server on port {} data-dir {}", port, data_dir);
            run_server(port, &data_dir).await
        }
        Commands::Client { host, port } => {
            println!("Connecting to {}:{}", host, port);
            run_repl(&host, port)
        }
    }
}

// --help and --version auto-generated by clap
"#;

println!("{}", main_code);

## 📝 Exercise: Add `--format` flag for JSON or plain text output

Add to Client: `#[arg(long, default_value = "plain")] format: String` — `plain` or `json`. When `json`, format responses as `{"ok": true, "value": "..."}` or `{"ok": false, "error": "..."}`.

In [ ]:
// YOUR CODE HERE
// Add --format to Client subcommand
// In REPL loop: when displaying response, if format == "json" then:
//   if response.starts_with("-ERR") { println!("{}", json!({ "ok": false, "error": response })); }
//   else { println!("{}", json!({ "ok": true, "value": response })); }
// Use serde_json::json! macro

println!("Implement --format flag for JSON or plain output.");

## 🎯 Key Takeaways

1. Use `clap` with `derive` for clean CLI: `#[derive(Parser)]`, `#[command]`, `#[arg]`
2. Subcommands: `server` and `client` with their own args
3. REPL: read line, send to server, read response, display — loop until QUIT
4. Colored output: green for success, red for errors — improves UX
5. Config priority: CLI args override env vars override config file override defaults

---
**Tomorrow:** Performance tuning